# T07 · Applications en science des données — ADIF72

**Objectifs :**
- Pipeline de modélisation prédictive complet
- Validation des modèles (k-fold, courbes d'apprentissage)
- Biais-variance, sur/sous-apprentissage
- Métriques d'évaluation (RMSE, R², précision, rappel, F1, AUC)
- Étude de cas end-to-end + biais & éthique

**Lien avec le reste :** ce topic assemble T01-T06 en pratique.
Toutes les notions (SVD→PCA, gradient→entraînement, tests→validation)
trouvent ici leur application.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     KFold, learning_curve)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                            precision_score, recall_score, f1_score,
                            roc_auc_score)
np.set_printoptions(precision=4, suppress=True, linewidth=100)
print("Prêt. sklearn importé.")
rng = np.random.default_rng(5)

## 1 · Pipeline de modélisation régressive (end-to-end)

Les **6 étapes** d'un pipeline classique :
1. **Données** : chargement, exploration
2. **Séparation** train/test (jamais mélanger !)
3. **Prétraitement** : échelle, encodage (fit SUR LE TRAIN SEUL)
4. **Entraînement** du modèle
5. **Évaluation** sur le test (une seule fois)
6. **Déploiement / itération**

**Piège n°1 : data leakage** — le prétraitement (ex. `StandardScaler`)
doit être `fit` sur les données train et `transform` sur le test.
Sinon le modèle « voit » la moyenne du test → évaluation trop optimiste.
En production, on encapsule prétraitement+modèle dans un `Pipeline`.

In [ ]:
n = 500
superficie = rng.uniform(50, 200, n)
age = rng.integers(0, 50, n)
nb_pieces = rng.integers(2, 7, n)
prix = (20000 * superficie - 1500 * age + 8000 * nb_pieces
        + rng.normal(0, 30000, n))
X = pd.DataFrame({'superficie': superficie, 'age': age, 'nb_pieces': nb_pieces})
y = pd.Series(prix)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=5)
scaler = StandardScaler().fit(X_tr)          # fit SUR LE TRAIN
X_tr_s = scaler.transform(X_tr); X_te_s = scaler.transform(X_te)

model = LinearRegression().fit(X_tr_s, y_tr)
pred = model.predict(X_te_s)
rmse = np.sqrt(mean_squared_error(y_te, pred))
r2 = r2_score(y_te, pred)
print(f"RMSE = {rmse:,.0f}  (bruit ~30k)")
print(f"R²   = {r2:.4f}")
assert r2 > 0.99 and rmse < 40000

### Exercice corrigé

In [ ]:
# ---- Exercice P1 : R^2 et RMSE a la main (donnees du pipeline) ----
y_pred = model.predict(X_te_s)
r2_man = 1 - np.sum((y_te - y_pred)**2) / np.sum((y_te - y_te.mean())**2)
rmse_man = np.sqrt(np.mean((y_te - y_pred)**2))
print(f"P1-ex : R^2 manuel = {r2_man:.4f}, RMSE = {rmse_man:,.0f}")
assert np.isclose(r2_man, r2_score(y_te, y_pred)) and r2_man > 0.99


## 2 · Validation k-fold & biais-variance

Le **train/test split** dépend de la chance de la coupe. Le **k-fold
cross-validation** est plus robuste : on coupe les données en `k` parts,
on entraîne `k` fois (à chaque fois sur `k-1` parts, test sur la restante),
puis on moyenne les scores.

**Courbe d'apprentissage** (learning curve) : on varie la taille du train
et on regarde l'écart train/validation.
- Écart grand → **sur-apprentissage** (variance)
- Deux courbes basses → **sous-apprentissage** (biais)
- Courbes qui convergent en haut → bon modèle

In [ ]:
# Données non linéaires (sin)
Xnl = rng.uniform(0, 1, (300, 1))
ynl = np.sin(6 * np.pi * Xnl[:, 0]) + rng.normal(0, 0.1, 300)

lin = LinearRegression()
forest = RandomForestRegressor(n_estimators=50, random_state=5)
for name, m in [("Linéaire", lin), ("Forêt", forest)]:
    kf = KFold(n_splits=5, shuffle=True, random_state=5)
    scores = cross_val_score(m, Xnl, ynl, cv=kf, scoring='r2')
    print(f"{name:8s} : R² moyen = {scores.mean():.3f} (±{scores.std():.3f})")

print("\nLe modèle trop simple (linéaire) sous-apprend sur du sin ;")
print("la forêt capture la non-linéarité (compromis biais/variance).")

### Exercice corrigé

In [ ]:
# ---- Exercice P2 : k-fold 5 plis sur sin ----
ex_rng = np.random.default_rng(101)
Xnl_ = ex_rng.uniform(0, 1, (300, 1))
ynl_ = np.sin(6*np.pi*Xnl_[:,0]) + ex_rng.normal(0, 0.1, 300)
sc5 = cross_val_score(LinearRegression(), Xnl_, ynl_, cv=5, scoring='r2')
print(f"P2-ex : R^2 (5 plis, lineaire) = {sc5.mean():.3f} (+/- {sc5.std():.3f}) -> sous-apprend")
assert sc5.mean() < 0.5


## 3 · Sur-apprentissage explicite (biais-variance)

Le **trade-off** : plus le modèle est flexible, moins il a de biais
mais plus il a de variance.
- **Biais** : erreur due aux hypothèses trop fortes du modèle (sous-apprend)
- **Variance** : sensibilité aux fluctuations de l'échantillon (sur-apprend)
- **MSE = biais² + variance + bruit**

On peut le visualiser avec un polynôme dont on augmente le degré :
au début le modèle est trop rigide (sous-apprend), puis il colle,
puis il épouse le bruit (sur-apprend).

In [ ]:
x = rng.uniform(-3, 3, 100)
y = np.cos(x) + rng.normal(0, 0.1, 100)
Xpoly = x.reshape(-1, 1)
Xtr, Xte, ytr, yte = train_test_split(Xpoly, y, test_size=0.3, random_state=5)

print("Degré -> R² train vs R² test :")
for deg in [1, 3, 8, 20]:
    poly = PolynomialFeatures(deg)
    Ptr = poly.fit_transform(Xtr); Pte = poly.transform(Xte)
    m = LinearRegression().fit(Ptr, ytr)
    r2_tr = m.score(Ptr, ytr); r2_te = m.score(Pte, yte)
    print(f"  degré {deg:2d} : train={r2_tr:.3f}  test={r2_te:.3f}"
          f"  {'<-- sur-apprend' if r2_tr - r2_te > 0.3 else ''}")

print("\nBas degré : sous-apprend (biais). Haut degré : sur-apprend (variance).")

### Exercice corrigé

In [ ]:
# ---- Exercice P3 : courbe de sur-apprentissage (degres 1, 3, 8, 20) ----
ex_rng = np.random.default_rng(102)
x_ = ex_rng.uniform(-3, 3, 100)
y_ = np.cos(x_) + ex_rng.normal(0, 0.1, 100)
Xp_ = x_.reshape(-1, 1)
Xtr_, Xte_, ytr_, yte_ = train_test_split(Xp_, y_, test_size=0.3, random_state=5)
for deg in [1, 3, 8, 20]:
    poly = PolynomialFeatures(deg)
    m = LinearRegression().fit(poly.fit_transform(Xtr_), ytr_)
    r2tr = m.score(poly.transform(Xtr_), ytr_)
    r2te = m.score(poly.transform(Xte_), yte_)
    print(f"P3-ex : degre {deg:2d} : train = {r2tr:.3f}  test = {r2te:.3f}")


## 4 · Classification : métriques

L'**accuracy** est trompeuse en présence de **déséquilibre** :
si 90% des cas sont négatifs, prédire toujours « négatif » donne
accuracy = 0.9 mais F1 ≈ 0. On utilise alors :
- **Précision** : P(déclaré positif | positif) — qualité des alertes
- **Rappel** : P(déclaré positif | vraiment positif) — sensibilité
- **F1** : moyenne harmonique précision/rappel
- **AUC-ROC** : qualité du classement (seuil-invariant) — la plus robuste

La **courbe ROC** trace rappel vs (1-précision) pour tous les seuils ;
AUC = aire sous la courbe.

In [ ]:
# Données équilibrées
Xc = rng.normal(0, 1, (400, 2))
Xc[:200] += np.array([2, 0]); Xc[200:] -= np.array([2, 0])
yc = np.array([1]*200 + [0]*200)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=5)
sc = StandardScaler().fit(Xc_tr)
clf = LogisticRegression(max_iter=1000).fit(sc.transform(Xc_tr), yc_tr)
pred = clf.predict(sc.transform(Xc_te))
prob = clf.predict_proba(sc.transform(Xc_te))[:, 1]
acc = accuracy_score(yc_te, pred); prec = precision_score(yc_te, pred)
rec = recall_score(yc_te, pred); f1 = f1_score(yc_te, pred)
auc = roc_auc_score(yc_te, prob)
print(f"Accuracy={acc:.3f}  Précision={prec:.3f}  Rappel={rec:.3f}  "
      f"F1={f1:.3f}  AUC={auc:.3f}")
assert acc > 0.95 and auc > 0.99

# Données déséquilibrées (90/10)
Xd = rng.normal(0, 1, (1000, 2)); yd = np.zeros(1000, int)
Xd[:100] += np.array([3, 0]); yd[:100] = 1
Xdt, Xde, ydt, yde = train_test_split(Xd, yd, test_size=0.3, random_state=5)
scd = StandardScaler().fit(Xdt)
clf2 = LogisticRegression(max_iter=1000).fit(scd.transform(Xdt), ydt)
pred2 = clf2.predict(scd.transform(Xde))
print(f"\nDéséquilibrées : Accuracy={accuracy_score(yde,pred2):.3f} "
      f"(trompeuse !)  F1={f1_score(yde,pred2,zero_division=0):.3f}")
print("-> préférer F1 / AUC à l'accuracy.")

### Exercice corrigé

In [ ]:
# ---- Exercice P4 : confusion matrix manuelle (12 cas, 4 positifs) ----
y4 = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0])
y4p = np.array([1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0])
TP = int(((y4 == 1) & (y4p == 1)).sum()); FP = int(((y4 == 0) & (y4p == 1)).sum())
FN = int(((y4 == 1) & (y4p == 0)).sum()); TN = int(((y4 == 0) & (y4p == 0)).sum())
acc = (TP + TN) / len(y4); prec = TP / (TP + FP); rec = TP / (TP + FN)
print(f"P4-ex : TP={TP} FP={FP} FN={FN} TN={TN}")
print(f"       accuracy = {acc:.4f}, precision = {prec:.4f}, rappel = {rec:.4f}")
assert abs(acc - 0.8333) < 1e-3 and abs(prec - 0.75) < 1e-3 and abs(rec - 0.75) < 1e-3


## 5 · Biais & considérations éthiques

Un modèle n'est que **aussi bon que ses données**. Les biais classiques :
- **Biais de sélection** : l'échantillon ne représente pas la population
  (ex : seulement les clients actifs). Le modèle est mauvais hors-échantillon.
- **Biais d'historique** : les données encodent les discriminations passées ;
  le modèle les reproduit.
- **Biais de mesure** : la variable cible est mal définie.

**Vérifications systématiques** avant de déployer :
- Couverture des variables de prédiction dans le jeu de test
- Équité (parité démographique, égalité des faux positifs par groupe)
- Sensibilité aux valeurs extrêmes

In [ ]:
# Biais de sélection : entraînement sur x>0 seulement
Xfull = rng.uniform(-5, 5, 2000)
yfull = np.sin(Xfull) + rng.normal(0, 0.1, 2000)
mask = Xfull > 0
Xb, yb = Xfull[mask].reshape(-1, 1), yfull[mask]
mb = LinearRegression().fit(Xb, yb)
err_biais = np.mean((mb.predict(Xfull[~mask].reshape(-1,1)) - yfull[~mask])**2)
print(f"Modèle biaisé (entraîné sur x>0) : coef={mb.coef_[0]:.3f}")
print(f"RMSE sur la partie non vue (x<0) = {np.sqrt(err_biais):.3f}")
print("Leçon : échantillon non représentatif = biais systématique.")

### Exercice corrigé

In [ ]:
# ---- Exercice P5 : biais d'un decideur (regle "selon groupe") ----
gr = np.array([0, 0, 0, 0, 1, 1, 1, 1])
dec = np.array([1, 1, 1, 1, 1, 1, 0, 0])
m0, m1 = dec[:4].mean(), dec[4:].mean()
print(f"P5-ex : taux de decision = {m0:.2f} (groupe 0) vs {m1:.2f} (groupe 1)")
print("       Ecart = 0.50 -> le decideur est biaise, meme si precis globalement")
assert m0 - m1 == 0.5


## 6 · Tableau récapitulatif — choisir la bonne méthode


| Situation | Méthode |
|---|---|
| n grande, x ~ normale | tests paramétriques (z, t) |
| n petit / non normal | tests non paramétriques |
| 3+ groupes (moyennes) | ANOVA + Tukey |
| indépendance (catégories) | chi-deux |
| prédiction continue | régression (lin, arbres, forêt) |
| prédiction binaire | régression logistique / arbres |
| données déséquilibrées | F1, AUC (pas l'accuracy) |
| sur-apprentissage | régularisation, plus de données |
| biais d'échantillonnage | stratification, vérif couverture |


In [ ]:
print("✅ T07 terminé.")

### Exercice corrigé

In [ ]:
# ---- Exercice P6 : lineaire vs arbre sur y = 3x + 2 (bruit faible) ----
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
ex_rng = np.random.default_rng(103)
X6 = ex_rng.uniform(0, 1, (200, 1))
y6 = 3 * X6[:,0] + 2 + ex_rng.normal(0, 0.05, 200)
Xtr, Xte, ytr, yte = train_test_split(X6, y6, test_size=0.3, random_state=1)
ml = LinearRegression().fit(Xtr, ytr)
mt = DecisionTreeRegressor().fit(Xtr, ytr)
rl, rt = ml.score(Xte, yte), mt.score(Xte, yte)
print(f"P6-ex : R2 test lineaire = {rl:.4f} ; R2 test arbre = {rt:.4f}")
print("       relation lineaire + bruit faible -> lineaire le mieux generalise")
assert rl > 0.99


## 7 · Synthèse — métriques de classification (F1, AUC)

### Exercice corrigé

In [ ]:
# ---- Exercice P7 : F1, precision, rappel, AUC a la main ----
TP, FP, FN, TN = 80, 10, 5, 105
P = TP/(TP+FP); R = TP/(TP+FN); F1 = 2*P*R/(P+R)
print(f"P7-ex : Precision = {P:.3f}, Rappel = {R:.3f}, F1 = {F1:.3f}")
assert np.isclose(F1, 0.9143, atol=1e-3)
y_ = np.array([1, 0, 1, 0, 1, 0, 1, 0])
s_ = np.array([0.9, 0.2, 0.8, 0.1, 0.7, 0.3, 0.4, 0.5])
pos = s_[y_==1]; neg = s_[y_==0]
wins = sum(1 for a in pos for b in neg if a > b)
ties = sum(1 for a in pos for b in neg if a == b)
auc = (wins + 0.5*ties) / (len(pos)*len(neg))
print(f"       AUC manuel = {auc:.4f}  (sklearn : {roc_auc_score(y_, s_):.4f})")
assert np.isclose(auc, 0.9375)
print("\n✅ T07 termine — 7 points, exercices integres.")
